# 🌿 Project 28 — Vegetation Change Detection

This is an important step because you're now moving from **monitoring vegetation** to **detecting vegetation change**.

You've already learned:

- NDVI → vegetation condition
- NDVI time series → vegetation change over time
- `.subtract()` → pixel-by-pixel difference

Now we'll combine these ideas to answer:

> **Where did vegetation increase, decrease, or remain relatively unchanged between two years?**

---

# 1. Introduce the Project

## Goal

Compare vegetation in the **same area** during two different years and create a change map showing:

- 🟢 Increased vegetation
- 🔴 Decreased vegetation
- ⚪ Little/no change

### Real-world applications

This type of analysis is used for:

- 🌳 Forest recovery
- 🌾 Agricultural change
- 🌵 Drought impacts
- 🔥 Post-fire vegetation recovery
- 🏜️ Land degradation
- 🌍 Ecosystem monitoring

---

# 2. Big Picture

Previously:

```text
One Image
    ↓
  NDVI
    ↓
Vegetation Map
````

Now:

```text
Year 1 Image
     ↓
   NDVI
     ↓
Year 2 Image
     ↓
   NDVI
     ↓
 Subtract
     ↓
NDVI Change
     ↓
Classify Change
     ↓
Increase / Decrease / No Change
```

The key equation is:

```text
NDVI Change = NDVI Year 2 − NDVI Year 1
```

---

# 3. What Does the Difference Mean?

Suppose one pixel has:

```text
2020 NDVI = 0.30
2025 NDVI = 0.70
```

Then:

```text
0.70 - 0.30 = +0.40
```

➡️ Vegetation increased.

Another pixel:

```text
2020 NDVI = 0.70
2025 NDVI = 0.30
```

```text
0.30 - 0.70 = -0.40
```

➡️ Vegetation decreased.

And:

```text
0.50 - 0.52 = -0.02
```

➡️ Very little change.

---

# 4. Step 1 — Define the Study Area

We'll use the same Berlin area.

```javascript
var studyArea = ee.Geometry.Polygon([
  [
    [13.20,52.35],
    [13.65,52.35],
    [13.65,52.65],
    [13.20,52.65],
    [13.20,52.35]
  ]
]);
```

---

# 5. Step 2 — Load Sentinel-2

We'll compare the same growing-season period in two years.

For example:

```text
2020 → June
2025 → June
```

Why use the same season?

Because vegetation naturally changes throughout the year.

Comparing June with December could confuse **seasonal change** with **long-term change**.

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);
```

---

# 6. Step 3 — Create a Year 1 Image

```javascript
var image2020 = collection
  .filterBounds(studyArea)
  .filterDate(
    '2020-06-01',
    '2020-06-30'
  )
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(studyArea);
```

The workflow is familiar:

```text
Collection
    ↓
filterBounds()
    ↓
filterDate()
    ↓
sort()
    ↓
first()
    ↓
Image
```

---

# 7. Step 4 — Create a Year 2 Image

```javascript
var image2025 = collection
  .filterBounds(studyArea)
  .filterDate(
    '2025-06-01',
    '2025-06-30'
  )
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(studyArea);
```

Now we have:

```text
image2020 → ee.Image
image2025 → ee.Image
```

---

# 8. Step 5 — Calculate NDVI

You've done this many times.

```javascript
var ndvi2020 = image2020
  .normalizedDifference(['B8','B4'])
  .rename('NDVI');

var ndvi2025 = image2025
  .normalizedDifference(['B8','B4'])
  .rename('NDVI');
```

Remember:

```text
B8 → NIR
B4 → Red
```

---

# 9. Step 6 — Calculate Vegetation Change ⭐

Now the important part:

```javascript
var vegetationChange = ndvi2025
  .subtract(ndvi2020);
```

This performs:

```text
Every pixel:

2025 NDVI - 2020 NDVI
```

The result is another `ee.Image`.

For example:

```text
Pixel A:

2020 = 0.2
2025 = 0.7

Change = +0.5
```

---

# 10. Step 7 — Create Change Categories

We don't just want the numerical difference.

We want three categories:

```text
Increase
Decrease
No Change
```

First, define a threshold.

```javascript
var threshold = 0.1;
```

Why not simply say:

```text
positive = increase
negative = decrease
```

Because tiny differences may come from:

* Sensor noise
* Atmospheric differences
* Soil conditions
* Remaining clouds
* Normal variability

So we'll consider only changes greater than **0.1** significant for this learning project.

---

# 11. Step 8 — Create Increased Vegetation

```javascript
var increased = vegetationChange.gt(threshold);
```

This asks:

> Is vegetation change greater than 0.1?

Example:

```text
+0.25 → true
+0.05 → false
-0.20 → false
```

---

# 12. Step 9 — Create Decreased Vegetation

```javascript
var decreased = vegetationChange.lt(-threshold);
```

This asks:

> Is vegetation change less than -0.1?

Example:

```text
-0.25 → true
-0.05 → false
+0.20 → false
```

Notice the `-threshold`.

That's important.

---

# 13. Step 10 — Create the Final Change Map ⭐

We can combine the three categories into one image.

```javascript
var changeMap = ee.Image(0)
  .where(
    vegetationChange.gt(threshold),
    1
  )
  .where(
    vegetationChange.lt(-threshold),
    -1
  );
```

Now:

```text
-1 → Decreased vegetation
 0 → No significant change
+1 → Increased vegetation
```

This is a **categorical change map**.

---

# 14. Step 11 — Visualize the Change

```javascript
var changeVis = {
  min: -1,
  max: 1,
  palette: [
    'red',
    'white',
    'green'
  ]
};
```

Then:

```javascript
Map.centerObject(
  studyArea,
  10
);

Map.addLayer(
  changeMap,
  changeVis,
  'Vegetation Change'
);
```

Interpretation:

```text
🔴 Red   → Decreased
⚪ White → No significant change
🟢 Green → Increased
```

---

# ⭐ Bonus — Display the Actual NDVI Difference

It's also useful to see the numerical change itself.

```javascript
var differenceVis = {
  min: -0.5,
  max: 0.5,
  palette: [
    'red',
    'white',
    'green'
  ]
};

Map.addLayer(
  vegetationChange,
  differenceVis,
  'NDVI Difference'
);
```

Now you can compare:

### Numerical map

```text
-0.5 → +0.5
```

with:

### Classified map

```text
-1 → 0 → +1
```

---

# Complete Code

```javascript
// 1. Study Area
var studyArea = ee.Geometry.Polygon([
  [
    [13.20,52.35],
    [13.65,52.35],
    [13.65,52.65],
    [13.20,52.65],
    [13.20,52.35]
  ]
]);


// 2. Sentinel-2 Collection
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);


// 3. Year 1
var image2020 = collection
  .filterBounds(studyArea)
  .filterDate(
    '2020-06-01',
    '2020-06-30'
  )
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(studyArea);


// 4. Year 2
var image2025 = collection
  .filterBounds(studyArea)
  .filterDate(
    '2025-06-01',
    '2025-06-30'
  )
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(studyArea);


// 5. NDVI
var ndvi2020 = image2020
  .normalizedDifference(['B8','B4'])
  .rename('NDVI');

var ndvi2025 = image2025
  .normalizedDifference(['B8','B4'])
  .rename('NDVI');


// 6. NDVI Change
var vegetationChange = ndvi2025
  .subtract(ndvi2020);


// 7. Threshold
var threshold = 0.1;


// 8. Increased Vegetation
var increased = vegetationChange
  .gt(threshold);


// 9. Decreased Vegetation
var decreased = vegetationChange
  .lt(-threshold);


// 10. Final Change Map
var changeMap = ee.Image(0)
  .where(
    vegetationChange.gt(threshold),
    1
  )
  .where(
    vegetationChange.lt(-threshold),
    -1
  );


// 11. Visualization
var changeVis = {
  min: -1,
  max: 1,
  palette: [
    'red',
    'white',
    'green'
  ]
};


// 12. Display
Map.centerObject(
  studyArea,
  10
);

Map.addLayer(
  changeMap,
  changeVis,
  'Vegetation Change'
);


// 13. Bonus: Numerical Difference
var differenceVis = {
  min: -0.5,
  max: 0.5,
  palette: [
    'red',
    'white',
    'green'
  ]
};

Map.addLayer(
  vegetationChange,
  differenceVis,
  'NDVI Difference'
);
```

---

# Things to Remember

## 1. `.subtract()`

You already learned this in Project 10.

```javascript
ndvi2025.subtract(ndvi2020)
```

means:

```text
Later - Earlier
```

The same concept works for:

* NDVI
* NDWI
* EVI
* Temperature
* Reflectance

---

## 2. Change Detection

The general pattern is:

```text
Before
  ↓
Calculate variable

After
  ↓
Calculate variable

     ↓

After - Before

     ↓

Change
```

This is one of the most important patterns in remote sensing.

---

## 3. Thresholding

We use:

```javascript
.gt(0.1)
```

and:

```javascript
.lt(-0.1)
```

to ignore very small changes.

But **0.1 is not a universal threshold**.

In real research, the threshold should be justified based on:

* Data quality
* Sensor
* Ecosystem
* Study objective
* Statistical analysis

---

# Common Beginner Mistakes

## ❌ Mistake 1: Comparing different seasons

For example:

```text
January 2020
```

vs.

```text
July 2025
```

The result would contain huge seasonal effects.

Better:

```text
June 2020
```

vs.

```text
June 2025
```

---

## ❌ Mistake 2: Reversing the subtraction

```javascript
ndvi2020.subtract(ndvi2025)
```

would reverse the meaning.

We want:

```javascript
ndvi2025.subtract(ndvi2020)
```

so:

```text
Positive → vegetation increased
Negative → vegetation decreased
```

---

## ❌ Mistake 3: Treating every tiny change as real change

A difference of:

```text
0.01
```

doesn't necessarily mean meaningful vegetation change.

That's why we introduced a threshold.

---

## ❌ Mistake 4: Using cloudy images

Clouds can produce artificial changes that look like vegetation loss or gain.

For a real study, use cloud masking and/or multi-image composites.

---

# Your Practice — Project 28.1

Create your own vegetation change analysis.

## Requirements

1. Choose a study area.
2. Choose two different years.
3. Use the **same season/month** for both.
4. Load Sentinel-2 imagery.
5. Calculate NDVI for both years.
6. Calculate NDVI difference.
7. Create three categories:

   * Increased vegetation
   * Decreased vegetation
   * No significant change
8. Display the final change map.

### Bonus

Try changing:

```javascript
var threshold = 0.1;
```

to:

```javascript
0.05
```

and:

```javascript
0.2
```

Then compare how the change map changes.

---

# Conceptual Questions

### Question 1

Why is it important to compare the **same season** in both years?

### Question 2

If:

```text
NDVI 2020 = 0.65
NDVI 2025 = 0.30
```

what is the NDVI change, and which category would it belong to if the threshold is `0.1`?

### Question 3

Why don't we simply classify every positive change as "increased vegetation" and every negative change as "decreased vegetation"?

